In [30]:
import os
import glob
import numpy as np
import pandas as pd
from PIL import Image
from collections import Counter
import shutil
from typing import Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

import sys
import os
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import albumentations as A
from albumentations.pytorch import ToTensorV2

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold

# Original dataset
path_train = "aptos/train_images"
label_train = "aptos/train.csv"


# Data split 85:15 
base_dir = 'data_split'
train_dir = os.path.join(base_dir, 'train_split')
test_dir = os.path.join(base_dir, 'test_split')
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# Plot style
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

In [14]:
# Other cofigs
DR_GRADES = {
    0: "No DR",
    1: "Mild",
    2: "Moderate", 
    3: "Severe",
    4: "Proliferative DR"
}

# Load and split data

In [7]:
# ----------- Data Loading -----------

def load_images(file_path):
    img_files = sorted(glob.glob(os.path.join(file_path, '*.png')))
    data = []
    for image_path in img_files:
        image = Image.open(image_path)
        data.append(np.array(image))
    return np.array(data), img_files

def copy_images(file_list, destination_folder, source_folder):
    for img_path in file_list:
        img_name = os.path.basename(img_path)
        src_path = os.path.join(source_folder, img_name)
        shutil.copy(src_path, destination_folder)

In [3]:
# Original dataset
path_train = "aptos/train_images"
label_train = "aptos/train.csv"

X, img_files = load_images(path_train)
y = pd.read_csv(label_train)

print(f"Total dataset size before split: {len(X)}")
assert len(X) == len(y), "Mismatch in images and labels"


Total dataset size before split: 3662


In [15]:
class_counts = y['diagnosis'].value_counts().sort_index()
class_pct = (class_counts / len(y) * 100).round(1)

print("Class Distribution:")
for grade, count in class_counts.items():
    name = DR_GRADES[grade]
    pct = class_pct[grade]
    print(f"Grade {grade} ({name:20s}): {count:5d}  ({pct:5.1f}%)")
print(f'  {"Total":26s}: {len(y):5d}')

max_count = class_counts.max()
min_count = class_counts.min()
print(f'Imbalance ratio (max/min): {max_count/min_count:.1f}x')

Class Distribution:
Grade 0 (No DR               ):  1805  ( 49.3%)
Grade 1 (Mild                ):   370  ( 10.1%)
Grade 2 (Moderate            ):   999  ( 27.3%)
Grade 3 (Severe              ):   193  (  5.3%)
Grade 4 (Proliferative DR    ):   295  (  8.1%)
  Total                     :  3662
Imbalance ratio (max/min): 9.4x


In [5]:
x_train, x_test, y_train, y_test, files_train, files_test = train_test_split(
    X, y, img_files, test_size=0.15, random_state=42, stratify=y['diagnosis'])

y_train.to_csv(os.path.join(base_dir, 'train_label.csv'), index=False)
y_test.to_csv(os.path.join(base_dir, 'test_label.csv'), index=False)

print("Class distribution before augmentation:")
print("Train:", Counter(y_train['diagnosis']))
print("Test:", Counter(y_test['diagnosis']))


Class distribution before augmentation:
Train: Counter({0: 1534, 2: 849, 1: 314, 4: 251, 3: 164})
Test: Counter({0: 271, 2: 150, 1: 56, 4: 44, 3: 29})


In [6]:
copy_images(files_train, train_dir, path_train)
copy_images(files_test, test_dir, path_train)

# Preprocessing + Augmentation

In [16]:
IMAGE_SIZE = 512
CROP_MARGIN = 0.05                   
BEN_GRAHAM_SIGMA_SCALE = 30  

In [ ]:
# preprocessing functions
def find_retina_circle(image: np.ndarray) -> Tuple[int, int, int]:
    #Detect the retina's bounding circle via thresholding + contour. -> (center_x, center_y, radius)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(
        thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    if not contours:
        h, w = image.shape[:2]
        return (w // 2, h // 2, min(w, h) // 2)

    largest = max(contours, key=cv2.contourArea)
    (x, y), radius = cv2.minEnclosingCircle(largest)
    return (int(x), int(y), int(radius))


def circular_crop(
    image: np.ndarray,
    target_size=IMAGE_SIZE,
    margin=CROP_MARGIN,
) -> np.ndarray:
    """Cắt ảnh theo hộp giới hạn của võng mạc + lề, resize"""
    h, w = image.shape[:2]
    cx, cy, r = find_retina_circle(image)
    r = int(r * (1 + margin))

    x1, y1 = max(0, cx - r), max(0, cy - r)
    x2, y2 = min(w, cx + r), min(h, cy + r)

    cropped = image[y1:y2, x1:x2]
    if cropped.shape[0] < 10 or cropped.shape[1] < 10:
        cropped = image

    resized = cv2.resize(
        cropped, (target_size, target_size), interpolation=cv2.INTER_LANCZOS4
    )
    mask = np.zeros((target_size, target_size), dtype=np.uint8)
    center = target_size // 2
    cv2.circle(mask, (center, center), center, 255, -1)
    resized[mask == 0] = 0
    return resized


#Step 2: local color normalization (gaussian blur weighted addition)
def local_color_normalization(
    image: np.ndarray,
    sigma_scale= BEN_GRAHAM_SIGMA_SCALE,
) -> np.ndarray:
    h, w = image.shape[:2]
    sigma = w / sigma_scale

    blurred = cv2.GaussianBlur(image, (0, 0), sigma)
    result = cv2.addWeighted(image, 4, blurred, -4, 128)

    return result


# Full pipeline
def ben_graham_preprocess(
    image: np.ndarray,
    target_size: int = IMAGE_SIZE,
) -> np.ndarray:
    cropped = circular_crop(image, target_size)
    result = local_color_normalization(cropped)
    return result


In [19]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

In [21]:
# Data Augmentation
def get_train_transform(image_size=IMAGE_SIZE):
    return A.Compose([
        A.Resize(image_size, image_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Affine(
            translate_percent={"x": (-0.1, 0.1), "y": (-0.1, 0.1)},
            scale=(0.9, 1.1),
            rotate=(-45, 45),
            mode=cv2.BORDER_CONSTANT,
            p=0.5,
        ),
        A.OneOf([
            A.RandomBrightnessContrast(
                brightness_limit=0.2, contrast_limit=0.2, p=1.0
            ),
            A.CLAHE(clip_limit=2.0, p=1.0),
        ], p=0.3),
        A.GaussNoise(std_range=(0.04, 0.20), p=0.2),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


def get_val_transform(image_size=IMAGE_SIZE):
    return A.Compose([
        A.Resize(image_size, image_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


In [45]:
class DRDataset(Dataset):
    def __init__(self, df: pd.DataFrame, image_dir, transform: A.Compose | None = None,preprocess: bool = True, target_size: int = IMAGE_SIZE):
        self.df = df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform
        self.preprocess = preprocess
        self.target_size = target_size

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        row = self.df.iloc[idx]
        image_id = str(row["id_code"])
        img_path = self.image_dir / f"{image_id}.png"

        image = cv2.imread(str(img_path))
        if image is None:
            raise ValueError(f"Failed to load: {img_path}")
            
        if self.preprocess:
            image = ben_graham_preprocess(image, self.target_size)

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        label = int(row["diagnosis"])

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented["image"]
        else:
            image = torch.from_numpy(
                image.transpose(2, 0, 1).copy()
            ).float() / 255.0

        return image, label

    def get_class_weights(self) -> torch.Tensor:
        """Compute inverse-frequency weights for ``FocalLoss`` alpha."""
        # Tinh trọng số nghịch đảo tần suất cho mỗi lớp
        counts = self.df["diagnosis"].value_counts().sort_index()
        total = len(self.df)
        weights = total / (len(counts) * counts)
        return torch.FloatTensor(weights.values)

## Split train/val

In [24]:
N_FOLDS = 5
RANDOM_SEED = 42

In [27]:
#Stratified K-Fold
def get_train_val_split(df,val_fold=0, n_folds=N_FOLDS, seed=RANDOM_SEED):
    """Stratified K-Fold split preserving class distribution."""
    df = df.copy()
    df["fold"] = -1

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for fold_idx, (_, val_idx) in enumerate(skf.split(df, df["diagnosis"])):
        df.loc[val_idx, "fold"] = fold_idx

    train_df = df[df["fold"] != val_fold].reset_index(drop=True)
    val_df = df[df["fold"] == val_fold].reset_index(drop=True)
    return train_df, val_df

In [51]:
train_df_CSV = pd.read_csv("data_split/train_label.csv")
train_df, val_df = get_train_val_split(train_df_CSV, val_fold=0)

print(f"Train fold size: {len(train_df)}")
print(f"Val fold size:   {len(val_df)}")
print(f"Class distribution:\n{train_df['diagnosis'].value_counts().sort_index()}")
# print(f"Class distribution:\n{val_df['diagnosis'].value_counts().sort_index()}")
display(train_df.head())

Train fold size: 2489
Val fold size:   623
Class distribution:
diagnosis
0    1227
1     251
2     679
3     131
4     201
Name: count, dtype: int64


,id_code,diagnosis,fold
0,025a169a0bb0,2,3
1,eeb231c3ef1f,1,1
2,a62e995f167c,0,4
3,f4ea2a2cfbb9,0,4
4,1c0cf251b426,1,2


In [46]:
BATCH_SIZE = 16
NUM_WORKERS = 4

In [55]:
train_dataset = DRDataset(df=train_df, image_dir=train_dir, transform=get_train_transform(IMAGE_SIZE), preprocess=ben_graham_preprocess, target_size=IMAGE_SIZE)
val_dataset = DRDataset(df=val_df, image_dir=train_dir, transform=get_val_transform(IMAGE_SIZE), preprocess=ben_graham_preprocess, target_size=IMAGE_SIZE)
train_loader = DataLoader(train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
)
val_loader = DataLoader(val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_34464\604007625.py:8: UserWarning: Argument(s) 'mode' are not valid for transform Affine
  A.Affine(


# Model

# Train/Val Functions

In [57]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


# Evaluate